## Лабораторная работа №1
### «Анализ и прогнозирование временных рядов на примере розничных продаж»

**Исходные данные:** `retail_sales_mock_data.csv` (месячные продажи + признаки `Promotion`, `HolidayMonth`).

**Цель:** провести EDA, декомпозицию временного ряда тремя подходами (классическая/FFT/вейвлет), построить прогнозные модели **ARIMA** и **SARIMAX**, оценить качество (MSE, R², AIC, BIC), выполнить анализ остатков и сделать вывод.

> Примечание: ноутбук рассчитан на запуск в Python 3.10+ (Anaconda/venv).

In [ ]:
# 1) Загрузка данных и предобработка

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import signal

from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from statsmodels.tsa.statespace.sarimax import SARIMAX

from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.stats.stattools import jarque_bera

from sklearn.metrics import mean_squared_error, r2_score

import pywt

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

DATA_PATH = "retail_sales_mock_data.csv"

In [ ]:
df = pd.read_csv(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values("Date").reset_index(drop=True)

# Индекс как временная шкала; частота начала месяца (MS)
df = df.set_index("Date")
df = df.asfreq("MS")

display(df.head(3))
print("shape:", df.shape)
print("date range:", df.index.min().date(), "→", df.index.max().date())
print("missing values:\n", df.isna().sum())

**Вывод (этап 1 — загрузка/предобработка):**
- Ряд имеет регулярную **месячную** частоту (MS), что удобно для сезонных моделей с периодом **12**.
- Пропусков по данным быть не должно (если они есть — их нужно обработать перед моделированием).

## 2.1 Разведочный анализ данных (EDA)

In [ ]:
y = df["SalesAmount"].astype(float)
exog = df[["Promotion", "HolidayMonth"]].astype(float)

# Базовые описательные статистики
eda_tbl = pd.DataFrame({
    "SalesAmount": y.describe(),
}).T

display(eda_tbl)

fig, ax = plt.subplots(1, 1)
y.plot(ax=ax, color="#2E86AB")
ax.set_title("Розничные продажи (временной ряд)")
ax.set_xlabel("Дата")
ax.set_ylabel("SalesAmount")
plt.show()

# Скользящее среднее/стд
roll12_mean = y.rolling(12).mean()
roll12_std = y.rolling(12).std()

fig, ax = plt.subplots(1, 1)
y.plot(ax=ax, alpha=0.6, label="sales")
roll12_mean.plot(ax=ax, label="rolling mean (12)")
roll12_std.plot(ax=ax, label="rolling std (12)")
ax.set_title("Скользящие характеристики (окно 12 месяцев)")
ax.legend()
plt.show()

# Распределение
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(y, kde=True, ax=ax[0], color="#2E86AB")
ax[0].set_title("Распределение SalesAmount")
sns.boxplot(x=y, ax=ax[1], color="#A23B72")
ax[1].set_title("Boxplot SalesAmount")
plt.show()

# Сезонность: продажи по месяцам года
month = y.index.month
month_names = pd.Index(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"], name="month")
by_month = pd.DataFrame({"SalesAmount": y.values, "month": month}).groupby("month")["SalesAmount"].agg(["mean", "median", "std"]).reindex(range(1, 13))
by_month.index = month_names

display(by_month)

fig, ax = plt.subplots(1, 1)
sns.boxplot(x=month, y=y.values, ax=ax)
ax.set_title("Сезонное распределение: продажи по месяцам")
ax.set_xlabel("Месяц")
ax.set_ylabel("SalesAmount")
plt.show()

# Экзогенные факторы: промо/праздники
fig, ax = plt.subplots(1, 1)
sns.scatterplot(x=exog["Promotion"], y=y, ax=ax)
ax.set_title("Связь продаж с Promotion")
ax.set_xlabel("Promotion (0/1)")
ax.set_ylabel("SalesAmount")
plt.show()

fig, ax = plt.subplots(1, 1)
sns.scatterplot(x=exog["HolidayMonth"], y=y, ax=ax, color="#A23B72")
ax.set_title("Связь продаж с HolidayMonth")
ax.set_xlabel("HolidayMonth (0/1)")
ax.set_ylabel("SalesAmount")
plt.show()

In [ ]:
# Проверка стационарности (ADF) + ACF/PACF

def adf_test(series, name="series"):
    s = pd.Series(series).dropna().astype(float)
    res = adfuller(s, autolag="AIC")
    out = {
        "name": name,
        "adf_stat": res[0],
        "p_value": res[1],
        "nobs": res[3],
    }
    return out

adf_raw = adf_test(y, "SalesAmount")
adf_diff1 = adf_test(y.diff(1), "diff1")
adf_seas_diff = adf_test(y.diff(12), "diff12")
adf_both = adf_test(y.diff(1).diff(12), "diff1+diff12")

adf_df = pd.DataFrame([adf_raw, adf_diff1, adf_seas_diff, adf_both])
display(adf_df)

fig, ax = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(y.dropna(), ax=ax[0], lags=24)
ax[0].set_title("ACF (исходный ряд)")
plot_pacf(y.dropna(), ax=ax[1], lags=24, method="ywm")
ax[1].set_title("PACF (исходный ряд)")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(y.diff(1).dropna(), ax=ax[0], lags=24)
ax[0].set_title("ACF (1-я разность)")
plot_pacf(y.diff(1).dropna(), ax=ax[1], lags=24, method="ywm")
ax[1].set_title("PACF (1-я разность)")
plt.tight_layout()
plt.show()

**Вывод (этап 2.1 — EDA):**
- По графику и сезонным boxplot обычно видно наличие **сезонности с периодом 12 месяцев** и возможного тренда.
- `Promotion` и `HolidayMonth` можно использовать как экзогенные регрессоры в `SARIMAX`.
- ADF-тест и ACF/PACF дают практическую подсказку по порядкам дифференцирования (обычно требуется хотя бы 1-я разность и/или сезонная разность).

## 2.1 Декомпозиция ряда (3 подхода)

In [ ]:
# (1) Классическая декомпозиция: аддитивная, мультипликативная, STL

period = 12

# Для мультипликативной декомпозиции нужны строго положительные значения
if (y <= 0).any():
    raise ValueError("Для мультипликативной декомпозиции SalesAmount должны быть > 0")

res_add = seasonal_decompose(y, model="additive", period=period, extrapolate_trend="freq")
res_mul = seasonal_decompose(y, model="multiplicative", period=period, extrapolate_trend="freq")
res_stl = STL(y, period=period, robust=True).fit()

fig = res_add.plot()
fig.set_size_inches(12, 8)
fig.suptitle("Классическая аддитивная декомпозиция", y=1.02)
plt.show()

fig = res_mul.plot()
fig.set_size_inches(12, 8)
fig.suptitle("Классическая мультипликативная декомпозиция", y=1.02)
plt.show()

fig = res_stl.plot()
fig.set_size_inches(12, 8)
fig.suptitle("STL-декомпозиция (robust=True)", y=1.02)
plt.show()

In [ ]:
# (2) Спектральный анализ: FFT
# Идея: найти доминирующие частоты (периоды). Для стационарности уберём тренд.

s = y.astype(float).copy()

# Детрэндинг (линейный) + удаление среднего
s_detr = signal.detrend(s.values)

n = len(s_detr)
# Частоты (циклов на шаг). Шаг = 1 месяц.
freqs = np.fft.rfftfreq(n, d=1.0)
fft_vals = np.fft.rfft(s_detr)
amplitude = np.abs(fft_vals)

# Уберём нулевую частоту (среднее)
mask = freqs > 0
freqs_nz = freqs[mask]
amp_nz = amplitude[mask]

# Период в месяцах = 1/f
periods = 1 / freqs_nz

# Отсортируем по амплитуде и посмотрим топ-периоды
order = np.argsort(amp_nz)[::-1]
top = pd.DataFrame({
    "frequency": freqs_nz[order][:10],
    "period_months": periods[order][:10],
    "amplitude": amp_nz[order][:10],
})
display(top)

fig, ax = plt.subplots(1, 1)
ax.stem(periods, amp_nz, basefmt=" ", linefmt="#2E86AB", markerfmt=" ")
ax.set_xlim(0, 30)
ax.set_title("Амплитудный спектр (FFT) в терминах периода (месяцы)")
ax.set_xlabel("Период, месяцев (1/f)")
ax.set_ylabel("Амплитуда")
ax.axvline(12, color="red", linestyle="--", alpha=0.6, label="12 месяцев")
ax.legend()
plt.show()

In [ ]:
# (3) Вейвлет-анализ: CWT (непрерывное) для оценки частот во времени
# Используем Морле (morl): получаем скалограмму (время x масштаб).

s_vals = y.astype(float).values

# Масштабы: чем больше масштаб, тем более "медленные" колебания (больше период)
scales = np.arange(1, 49)
wavelet = "morl"

coeffs, freqs_cwt = pywt.cwt(s_vals, scales=scales, wavelet=wavelet)

# Для удобства переведём "псевдочастоту" в период (в месяцах)
# sampling_period=1 месяц
pseudo_freq = pywt.scale2frequency(wavelet, scales) / 1.0
pseudo_period = 1 / pseudo_freq

power = np.abs(coeffs) ** 2

fig, ax = plt.subplots(1, 1, figsize=(14, 5))
# imshow: по оси Y — периоды (месяцы). Ограничим разумным диапазоном
extent = [0, len(y) - 1, pseudo_period.max(), pseudo_period.min()]
im = ax.imshow(power, extent=extent, cmap="viridis", aspect="auto")
ax.set_title("Скалограмма (CWT, Morlet): мощность по времени и периоду")
ax.set_xlabel("Индекс времени (месяцы)")
ax.set_ylabel("Период (месяцы)")

# Подпишем ось времени реальными датами (редко, чтобы не шумело)
ticks = np.linspace(0, len(y) - 1, 6).astype(int)
ax.set_xticks(ticks)
ax.set_xticklabels([y.index[i].strftime("%Y-%m") for i in ticks])

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Power")

ax.axhline(12, color="red", linestyle="--", alpha=0.7)
plt.show()

**Вывод (этап 2.1 — сравнение декомпозиций):**

- **Классическая (`seasonal_decompose`)**
  - **Плюсы**: проста, быстро даёт интерпретируемые компоненты (trend/seasonal/resid).
  - **Минусы**: чувствительна к выбросам и к тому, что сезонность меняется со временем; требует корректного `period`.
  - **Аддитивная vs мультипликативная**: мультипликативная полезна, когда амплитуда сезонности растёт вместе с уровнем ряда.

- **STL**
  - **Плюсы**: более гибкая, лучше работает при шуме/выбросах (особенно `robust=True`), допускает более “плавную” сезонность.
  - **Минусы**: тоже требует выбора периода; интерпретация параметров менее прямолинейна.

- **FFT (спектральный анализ)**
  - **Плюсы**: хорошо находит доминирующие периоды (например, около 12 месяцев), полезен для первичной проверки сезонности.
  - **Минусы**: предполагает стационарность/квази-стационарность; плохо показывает, **как** сезонность менялась во времени.

- **Вейвлет (CWT)**
  - **Плюсы**: показывает, **когда** во времени проявляются колебания заданных периодов; подходит при нестационарной сезонности.
  - **Минусы**: больше настроек (вейвлет/масштабы), сложнее интерпретация, результаты чувствительны к краевым эффектам.

Практически: для построения SARIMAX полезнее всего классическая/СТL (для понимания тренда/сезонности), FFT — для подтверждения периода, вейвлет — для проверки нестационарности сезонных компонент.

## 2.2 Построение прогнозных моделей (ARIMA и SARIMAX)

In [ ]:
# Разделение на train/test
# При 48 точках разумно оставить последние 12 месяцев на тест.

test_size = 12
train_y, test_y = y.iloc[:-test_size], y.iloc[-test_size:]
train_exog, test_exog = exog.iloc[:-test_size], exog.iloc[-test_size:]

print("train:", train_y.index.min().date(), "→", train_y.index.max().date(), "n=", len(train_y))
print("test :", test_y.index.min().date(), "→", test_y.index.max().date(), "n=", len(test_y))

# Горизонт прогноза: максимально возможный на test
h = len(test_y)
assert h == test_size

In [ ]:
# Подбор параметров по AIC: компактный grid search

from itertools import product


def fit_sarimax(endog, exog=None, order=(1,0,0), seasonal_order=(0,0,0,0), trend="n"):
    model = SARIMAX(
        endog,
        exog=exog,
        order=order,
        seasonal_order=seasonal_order,
        trend=trend,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    res = model.fit(disp=False)
    return res


def grid_search_aic(
    endog,
    exog=None,
    p_range=(0, 1, 2),
    d_range=(0, 1),
    q_range=(0, 1, 2),
    P_range=(0, 1),
    D_range=(0, 1),
    Q_range=(0, 1),
    s=12,
    seasonal=True,
    trend="n",
    max_evals=200,
):
    rows = []
    evals = 0

    for (p, d, q) in product(p_range, d_range, q_range):
        if seasonal:
            for (P, D, Q) in product(P_range, D_range, Q_range):
                seasonal_order = (P, D, Q, s)
                try:
                    res = fit_sarimax(endog, exog=exog, order=(p, d, q), seasonal_order=seasonal_order, trend=trend)
                    rows.append({
                        "order": (p, d, q),
                        "seasonal_order": seasonal_order,
                        "aic": res.aic,
                        "bic": res.bic,
                    })
                except Exception:
                    continue
                evals += 1
                if evals >= max_evals:
                    break
            if evals >= max_evals:
                break
        else:
            try:
                res = fit_sarimax(endog, exog=exog, order=(p, d, q), seasonal_order=(0, 0, 0, 0), trend=trend)
                rows.append({
                    "order": (p, d, q),
                    "seasonal_order": (0, 0, 0, 0),
                    "aic": res.aic,
                    "bic": res.bic,
                })
            except Exception:
                continue
            evals += 1
            if evals >= max_evals:
                break

    out = pd.DataFrame(rows).sort_values(["aic", "bic"], ascending=True).reset_index(drop=True)
    return out

In [ ]:
# 1) ARIMA (без сезонности и без экзогенных факторов) — базовая модель
# Поскольку данных мало, ограничимся небольшими диапазонами.

arima_candidates = grid_search_aic(
    train_y,
    exog=None,
    p_range=(0, 1, 2, 3),
    d_range=(0, 1),
    q_range=(0, 1, 2, 3),
    seasonal=False,
    trend="n",
    max_evals=300,
)

display(arima_candidates.head(10))

best_arima_order = tuple(arima_candidates.loc[0, "order"])
print("Best ARIMA order:", best_arima_order)

arima_res = fit_sarimax(train_y, exog=None, order=best_arima_order, seasonal_order=(0, 0, 0, 0), trend="n")
print(arima_res.summary())

In [ ]:
# 2) SARIMAX (сезонность 12 + экзогенные факторы)

sarimax_candidates = grid_search_aic(
    train_y,
    exog=train_exog,
    p_range=(0, 1, 2),
    d_range=(0, 1),
    q_range=(0, 1, 2),
    P_range=(0, 1),
    D_range=(0, 1),
    Q_range=(0, 1),
    s=12,
    seasonal=True,
    trend="n",
    max_evals=300,
)

display(sarimax_candidates.head(10))

best_sarimax_order = tuple(sarimax_candidates.loc[0, "order"])
best_sarimax_seasonal = tuple(sarimax_candidates.loc[0, "seasonal_order"])
print("Best SARIMAX order:", best_sarimax_order)
print("Best SARIMAX seasonal_order:", best_sarimax_seasonal)

sarimax_res = fit_sarimax(train_y, exog=train_exog, order=best_sarimax_order, seasonal_order=best_sarimax_seasonal, trend="n")
print(sarimax_res.summary())

In [ ]:
# Прогноз на горизонт test (12 месяцев)

# ARIMA прогноз
arima_fc = arima_res.get_forecast(steps=h)
arima_mean = arima_fc.predicted_mean
arima_ci = arima_fc.conf_int(alpha=0.05)

# SARIMAX прогноз (нужны exog на период прогноза)
sarimax_fc = sarimax_res.get_forecast(steps=h, exog=test_exog)
sarimax_mean = sarimax_fc.predicted_mean
sarimax_ci = sarimax_fc.conf_int(alpha=0.05)

# Визуализация
fig, ax = plt.subplots(1, 1, figsize=(14, 5))
train_y.plot(ax=ax, label="train", color="#2E86AB")
test_y.plot(ax=ax, label="test", color="black")

arima_mean.index = test_y.index
sarimax_mean.index = test_y.index

ax.plot(test_y.index, arima_mean.values, label=f"ARIMA{best_arima_order}", color="#F18F01")
ax.fill_between(test_y.index, arima_ci.iloc[:, 0].values, arima_ci.iloc[:, 1].values, color="#F18F01", alpha=0.15)

ax.plot(test_y.index, sarimax_mean.values, label=f"SARIMAX{best_sarimax_order}x{best_sarimax_seasonal}", color="#A23B72")
ax.fill_between(test_y.index, sarimax_ci.iloc[:, 0].values, sarimax_ci.iloc[:, 1].values, color="#A23B72", alpha=0.12)

ax.set_title("Прогноз на 12 месяцев: ARIMA vs SARIMAX")
ax.set_xlabel("Дата")
ax.set_ylabel("SalesAmount")
ax.legend()
plt.show()

**Вывод (этап 2.2 — модели):**
- `ARIMA` используется как базовая модель только по самому ряду.
- `SARIMAX` учитывает **сезонность (12)** и экзогенные факторы (`Promotion`, `HolidayMonth`), поэтому при наличии устойчивой сезонности и влияния факторов часто даёт лучший прогноз.
- Подбор параметров выполнен через минимизацию **AIC** на ограниченной сетке (компромисс между качеством и малым объёмом данных).

## 2.3 Оценка качества моделей и анализ остатков

In [ ]:
# Метрики на тесте

def eval_forecast(y_true, y_pred, name, aic, bic):
    y_true = pd.Series(y_true).astype(float)
    y_pred = pd.Series(y_pred).astype(float)
    return {
        "model": name,
        "MSE": mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "AIC": float(aic),
        "BIC": float(bic),
    }

arima_metrics = eval_forecast(
    test_y,
    arima_mean.values,
    name=f"ARIMA{best_arima_order}",
    aic=arima_res.aic,
    bic=arima_res.bic,
)

sarimax_metrics = eval_forecast(
    test_y,
    sarimax_mean.values,
    name=f"SARIMAX{best_sarimax_order}x{best_sarimax_seasonal} + exog",
    aic=sarimax_res.aic,
    bic=sarimax_res.bic,
)

metrics_df = pd.DataFrame([arima_metrics, sarimax_metrics]).sort_values("MSE")
display(metrics_df)

# Ошибка по времени
err = pd.DataFrame({
    "y_true": test_y.values,
    "ARIMA": test_y.values - arima_mean.values,
    "SARIMAX": test_y.values - sarimax_mean.values,
}, index=test_y.index)

fig, ax = plt.subplots(1, 1, figsize=(14, 4))
err[["ARIMA", "SARIMAX"]].plot(ax=ax)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Ошибки прогноза на тестовом периоде (y_true - y_pred)")
ax.set_xlabel("Дата")
plt.show()

In [ ]:
# Диагностика остатков

import statsmodels.api as sm


def residual_diagnostics(res, title):
    resid = pd.Series(res.resid).dropna().astype(float)

    jb_stat, jb_p, skew, kurt = jarque_bera(resid)
    lb = acorr_ljungbox(resid, lags=[12], return_df=True)
    arch_stat, arch_p, _, _ = het_arch(resid)

    out = {
        "model": title,
        "JB_p": float(jb_p),
        "LjungBox_p(lag12)": float(lb["lb_pvalue"].iloc[0]),
        "ARCH_p": float(arch_p),
    }

    fig, ax = plt.subplots(2, 2, figsize=(14, 8))

    ax[0, 0].plot(resid.index, resid.values)
    ax[0, 0].axhline(0, color="black", linewidth=1)
    ax[0, 0].set_title(f"Остатки во времени: {title}")

    sns.histplot(resid, kde=True, ax=ax[0, 1], color="#2E86AB")
    ax[0, 1].set_title("Распределение остатков")

    sm.qqplot(resid, line="45", ax=ax[1, 0])
    ax[1, 0].set_title("Q-Q plot")

    plot_acf(resid, ax=ax[1, 1], lags=24)
    ax[1, 1].set_title("ACF остатков")

    plt.tight_layout()
    plt.show()

    return out

arima_diag = residual_diagnostics(arima_res, f"ARIMA{best_arima_order}")
sarimax_diag = residual_diagnostics(sarimax_res, f"SARIMAX{best_sarimax_order}x{best_sarimax_seasonal} + exog")

diag_df = pd.DataFrame([arima_diag, sarimax_diag])
display(diag_df)

print("Интерпретация p-values:")
print("- JB_p (нормальность): p>0.05 → нет оснований отвергать нормальность")
print("- LjungBox_p: p>0.05 → нет значимой автокорреляции в остатках")
print("- ARCH_p: p>0.05 → нет выраженной условной гетероскедастичности")

**Вывод (этап 2.3 — качество и остатки):**
- Основная метрика качества прогноза на отложенном периоде — **MSE** (меньше лучше) и **R²** (больше лучше).
- **AIC/BIC** сравнивают модели с учётом сложности: меньше — лучше (но сравнивать корректно модели, обученные на одном и том же train).
- Хорошая модель обычно даёт остатки, похожие на “белый шум”: без автокорреляции (Ljung–Box), без сильной гетероскедастичности (ARCH), с приемлемой нормальностью (JB) — хотя в прикладных задачах строгая нормальность не всегда обязательна.
- Предпочтительная модель выбирается по совокупности: качество прогноза + критерии + адекватность остатков.

## 2.4 Документирование и интерпретация (итог)

### Итоговый вывод

- По результатам EDA и декомпозиции ряд выглядит **сезонным (≈12 месяцев)**; есть смысл использовать сезонную модель.
- Если экзогенные факторы действительно влияют на уровень продаж, `SARIMAX` обычно выигрывает у `ARIMA` по **MSE/R²** на тесте и даёт более “здоровые” остатки.
- Если разница в качестве между моделями небольшая, приоритет обычно у модели с:
  - меньшими **AIC/BIC** (при обучении на одном train),
  - менее автокоррелированными остатками,
  - более стабильным поведением ошибок.

> В этом ноутбуке предпочтительная модель определяется по таблице `metrics_df` и диагностике `diag_df`.